In [ ]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Batch contradiction agent for healthcare facilities
# MAGIC
# MAGIC This notebook processes one or more facilities, detects contradictions,
# MAGIC maps them to a standard ontology, and writes results to Delta tables.
# COMMAND ----------
# Optional if you are using UC functions through the Databricks Function Client
# %pip install unitycatalog-ai[databricks]
# dbutils.library.restartPython()

# COMMAND ----------
import json
import math
import uuid
from datetime import datetime
from typing import Any, Dict, List, Optional

from pyspark.sql import functions as F
from pyspark.sql import types as T
# COMMAND ----------
CATALOG = "datalake_dev"
BRONZE_SCHEMA = "virtue_foundation_dataset"
GOV_SCHEMA = "governance"
RESULTS_SCHEMA = "results"
ORCH_SCHEMA = "orchestration"
TOOLS_SCHEMA = "tools"

RAW_FACILITY_TABLE = f"{CATALOG}.{BRONZE_SCHEMA}.facilities"
PINCODE_TABLE = f"{CATALOG}.{BRONZE_SCHEMA}.india_post_pincode_directory"

CONTRADICTION_ONTOLOGY_TABLE = f"{CATALOG}.{GOV_SCHEMA}.contradiction_ontology"
CONTRADICTION_TAGS_TABLE = f"{CATALOG}.{GOV_SCHEMA}.contradiction_tags"
CAPABILITY_ONTOLOGY_TABLE = f"{CATALOG}.{GOV_SCHEMA}.facility_capability_ontology"

RUN_SUMMARY_TABLE = f"{CATALOG}.{RESULTS_SCHEMA}.facility_contradiction_runs"
DETAIL_TABLE = f"{CATALOG}.{RESULTS_SCHEMA}.facility_contradictions"
EVIDENCE_TABLE = f"{CATALOG}.{RESULTS_SCHEMA}.facility_contradiction_evidence"

QUEUE_TABLE = f"{CATALOG}.{ORCH_SCHEMA}.facility_processing_queue"

AGENT_VERSION = "contradiction-agent-v1"
ONTOLOGY_VERSION = "ontology-v1"
MAX_BATCH_SIZE = 500
DISTANCE_THRESHOLD_KM = 50.0
CURRENT_YEAR = datetime.utcnow().year
# COMMAND ----------
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{GOV_SCHEMA}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{RESULTS_SCHEMA}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{ORCH_SCHEMA}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{TOOLS_SCHEMA}")
# COMMAND ----------
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CONTRADICTION_ONTOLOGY_TABLE} (
  contradiction_code STRING,
  contradiction_class STRING,     -- HARD / LOGICAL / WEAK_EVIDENCE
  contradiction_name STRING,
  severity_default STRING,        -- HIGH / MEDIUM / LOW
  description STRING,
  review_guidance STRING,
  active BOOLEAN
) USING DELTA
""")

# COMMAND ----------
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CONTRADICTION_TAGS_TABLE} (
  contradiction_code STRING,
  tag STRING
) USING DELTA
""")
# COMMAND ----------
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CAPABILITY_ONTOLOGY_TABLE} (
  concept_id STRING,
  concept_name STRING,
  concept_type STRING,            -- SPECIALTY / EQUIPMENT / PROCEDURE / CAPABILITY
  parent_concept_id STRING,
  synonyms ARRAY<STRING>,
  active BOOLEAN
) USING DELTA
""")
# COMMAND ----------
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {RUN_SUMMARY_TABLE} (
  run_id STRING,
  facility_id STRING,
  run_ts TIMESTAMP,
  agent_version STRING,

  hard_count INT,
  logical_count INT,
  weak_evidence_count INT,

  overall_risk_score DOUBLE,
  review_priority STRING,

  ontology_version STRING,
  processing_status STRING,
  error_message STRING
) USING DELTA
""")
# COMMAND ----------
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {DETAIL_TABLE} (
  run_id STRING,
  facility_id STRING,
  contradiction_id STRING,
  contradiction_code STRING,
  contradiction_class STRING,
  severity STRING,
  confidence DOUBLE,

  concept_ids ARRAY<STRING>,
  tags ARRAY<STRING>,

  explanation STRING,
  recommended_action STRING,
  created_ts TIMESTAMP
) USING DELTA
""")
# COMMAND ----------
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {EVIDENCE_TABLE} (
  run_id STRING,
  facility_id STRING,
  contradiction_id STRING,
  evidence_type STRING,      -- STRUCTURED_FIELD / DESCRIPTION_QUOTE / LOOKUP_MATCH / SOURCE_URL
  evidence_field STRING,
  evidence_value STRING,
  evidence_quote STRING,
  source_ref STRING,
  created_ts TIMESTAMP
) USING DELTA
""")

# COMMAND ----------
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {QUEUE_TABLE} (
  facility_id STRING,
  status STRING,             -- PENDING / RUNNING / SUCCESS / FAILED
  last_run_id STRING,
  attempts INT,
  updated_ts TIMESTAMP
) USING DELTA
""")
# COMMAND ----------
ontology_rows = [
    ("GEO_STATE_MISMATCH", "HARD", "Postcode-state mismatch", "HIGH",
     "Facility state disagrees with postcode directory state.",
     "Review address and postcode; correct source or coordinates.", True),

    ("GEO_POSTCODE_COORD_MISMATCH", "HARD", "Postcode-coordinate mismatch", "HIGH",
     "Facility coordinates are too far from the postcode location.",
     "Verify coordinates and postcode.", True),

    ("INVALID_YEAR", "HARD", "Invalid establishment year", "HIGH",
     "Year established is not parseable or is in the future.",
     "Correct yearEstablished or mark as unknown.", True),

    ("EXPLICIT_NEGATION_CONFLICT", "HARD", "Explicit text negation conflict", "HIGH",
     "Structured fields claim a capability but the description explicitly negates it.",
     "Escalate for human review.", True),

    ("DUPLICATE_ENTITY_CONFLICT", "HARD", "Likely duplicate facility conflict", "HIGH",
     "Entity appears duplicated with conflicting key attributes.",
     "Review cluster and merge logic.", True),

    ("STRUCTURED_TEXT_MISMATCH", "LOGICAL", "Structured-text mismatch", "MEDIUM",
     "Structured fields and text evidence do not align.",
     "Review evidence and override if needed.", True),

    ("EQUIPMENT_SPECIALTY_MISMATCH", "LOGICAL", "Equipment-specialty mismatch", "MEDIUM",
     "Equipment implies a specialty not represented in structured data.",
     "Review specialty mapping.", True),

    ("PROCEDURE_CAPABILITY_MISMATCH", "LOGICAL", "Procedure-capability mismatch", "MEDIUM",
     "Procedure implies a missing capability or specialty.",
     "Review procedures/capabilities.", True),

    ("STAFFING_CAPABILITY_MISMATCH", "LOGICAL", "Staffing-capability mismatch", "MEDIUM",
     "Claimed capabilities seem inconsistent with staffing/capacity fields.",
     "Review numberDoctors/capacity evidence.", True),

    ("UNSUPPORTED_STRUCTURED_CLAIM", "WEAK_EVIDENCE", "Unsupported structured claim", "MEDIUM",
     "Structured claim lacks support in free text.",
     "Seek source evidence or confirm manually.", True),

    ("VAGUE_MARKETING_CLAIM", "WEAK_EVIDENCE", "Vague marketing claim", "LOW",
     "Text uses promotional language without concrete evidence.",
     "Treat as low-confidence until verified.", True),

    ("SOURCE_CONFLICT", "WEAK_EVIDENCE", "Source conflict", "MEDIUM",
     "Source-derived claims disagree or are weakly supported.",
     "Check official website or source URLs.", True),

    ("STALE_INFORMATION", "WEAK_EVIDENCE", "Stale information", "LOW",
     "Record appears outdated based on freshness indicators.",
     "Treat as lower-confidence and consider revalidation.", True),
]
# COMMAND ----------
ontology_schema = T.StructType([
    T.StructField("contradiction_code", T.StringType()),
    T.StructField("contradiction_class", T.StringType()),
    T.StructField("contradiction_name", T.StringType()),
    T.StructField("severity_default", T.StringType()),
    T.StructField("description", T.StringType()),
    T.StructField("review_guidance", T.StringType()),
    T.StructField("active", T.BooleanType()),
])

ontology_df = spark.createDataFrame(ontology_rows, ontology_schema)
ontology_df.write.mode("overwrite").saveAsTable(CONTRADICTION_ONTOLOGY_TABLE)
# COMMAND ----------
tag_rows = [
    ("GEO_STATE_MISMATCH", "geo"),
    ("GEO_STATE_MISMATCH", "postcode"),
    ("GEO_STATE_MISMATCH", "state"),

    ("GEO_POSTCODE_COORD_MISMATCH", "geo"),
    ("GEO_POSTCODE_COORD_MISMATCH", "coordinates"),
    ("GEO_POSTCODE_COORD_MISMATCH", "postcode"),

    ("INVALID_YEAR", "metadata"),
    ("INVALID_YEAR", "year"),

    ("EXPLICIT_NEGATION_CONFLICT", "description"),
    ("EXPLICIT_NEGATION_CONFLICT", "capability"),
    ("EXPLICIT_NEGATION_CONFLICT", "high_conflict"),

    ("STRUCTURED_TEXT_MISMATCH", "specialty"),
    ("STRUCTURED_TEXT_MISMATCH", "description"),
    ("STRUCTURED_TEXT_MISMATCH", "needs_human_review"),

    ("EQUIPMENT_SPECIALTY_MISMATCH", "equipment"),
    ("EQUIPMENT_SPECIALTY_MISMATCH", "specialty"),

    ("PROCEDURE_CAPABILITY_MISMATCH", "procedure"),
    ("PROCEDURE_CAPABILITY_MISMATCH", "capability"),

    ("STAFFING_CAPABILITY_MISMATCH", "staffing"),
    ("STAFFING_CAPABILITY_MISMATCH", "capacity"),

    ("UNSUPPORTED_STRUCTURED_CLAIM", "weak_evidence"),
    ("UNSUPPORTED_STRUCTURED_CLAIM", "structured_field"),

    ("VAGUE_MARKETING_CLAIM", "weak_evidence"),
    ("VAGUE_MARKETING_CLAIM", "marketing_claim"),

    ("SOURCE_CONFLICT", "source"),
    ("SOURCE_CONFLICT", "weak_evidence"),

    ("STALE_INFORMATION", "freshness"),
]
# COMMAND ----------
tag_schema = T.StructType([
    T.StructField("contradiction_code", T.StringType()),
    T.StructField("tag", T.StringType()),
])

tag_df = spark.createDataFrame(tag_rows, tag_schema)
tag_df.write.mode("overwrite").saveAsTable(CONTRADICTION_TAGS_TABLE)
# COMMAND ----------
cap_rows = [
    ("SPEC_CARDIOLOGY", "Cardiology", "SPECIALTY", None, ["cardiac", "heart care"], True),
    ("SPEC_ONCOLOGY", "Oncology", "SPECIALTY", None, ["cancer care", "chemotherapy"], True),
    ("SPEC_RADIOLOGY", "Radiology", "SPECIALTY", None, ["imaging", "diagnostics"], True),
    ("SPEC_OBSTETRICS", "Obstetrics", "SPECIALTY", None, ["maternity", "obgyn", "obstetric"], True),

    ("EQ_MRI", "MRI", "EQUIPMENT", None, ["mri machine", "magnetic resonance"], True),
    ("EQ_CT", "CT Scanner", "EQUIPMENT", None, ["ct scan", "ct scanner"], True),
    ("EQ_VENTILATOR", "Ventilator", "EQUIPMENT", None, ["ventilator"], True),

    ("PROC_C_SECTION", "Caesarean section", "PROCEDURE", None, ["c-section", "cesarean"], True),
    ("PROC_CHEMOTHERAPY", "Chemotherapy", "PROCEDURE", None, ["chemo"], True),

    ("CAP_ICU", "Intensive Care Unit", "CAPABILITY", None, ["icu", "intensive care"], True),
    ("CAP_EMERGENCY_24X7", "24x7 Emergency", "CAPABILITY", None, ["24x7 emergency", "emergency care"], True),
]
# COMMAND ----------
cap_schema = T.StructType([
    T.StructField("concept_id", T.StringType()),
    T.StructField("concept_name", T.StringType()),
    T.StructField("concept_type", T.StringType()),
    T.StructField("parent_concept_id", T.StringType()),
    T.StructField("synonyms", T.ArrayType(T.StringType())),
    T.StructField("active", T.BooleanType()),
])

cap_df = spark.createDataFrame(cap_rows, cap_schema)
cap_df.write.mode("overwrite").saveAsTable(CAPABILITY_ONTOLOGY_TABLE)
# COMMAND ----------
spark.sql(f"""
CREATE OR REPLACE FUNCTION {CATALOG}.{TOOLS_SCHEMA}.lookup_pincode(pin STRING)
RETURNS TABLE (
  pincode BIGINT,
  district STRING,
  statename STRING,
  latitude STRING,
  longitude STRING
)
RETURN
SELECT pincode, district, statename, latitude, longitude
FROM {PINCODE_TABLE}
WHERE CAST(pincode AS STRING) = pin
""")
# COMMAND ----------
spark.sql(f"""
CREATE OR REPLACE FUNCTION {CATALOG}.{TOOLS_SCHEMA}.lookup_contradiction_tags(code STRING)
RETURNS TABLE (
  contradiction_code STRING,
  tag STRING
)
RETURN
SELECT contradiction_code, tag
FROM {CONTRADICTION_TAGS_TABLE}
WHERE contradiction_code = code
""")
# COMMAND ----------
contradiction_ontology_df = spark.table(CONTRADICTION_ONTOLOGY_TABLE)
contradiction_tags_df = spark.table(CONTRADICTION_TAGS_TABLE)
capability_ontology_df = spark.table(CAPABILITY_ONTOLOGY_TABLE)

contradiction_ontology = {
    row["contradiction_code"]: row.asDict()
    for row in contradiction_ontology_df.collect()
}

contradiction_tags = {}
for row in contradiction_tags_df.collect():
    contradiction_tags.setdefault(row["contradiction_code"], []).append(row["tag"])

capability_rows = capability_ontology_df.collect()
# COMMAND ----------
def haversine_km(lat1, lon1, lat2, lon2):
    if None in [lat1, lon1, lat2, lon2]:
        return None
    r = 6371.0
    p = math.pi / 180.0
    dlat = (lat2 - lat1) * p
    dlon = (lon2 - lon1) * p
    a = math.sin(dlat/2)**2 + math.cos(lat1*p) * math.cos(lat2*p) * math.sin(dlon/2)**2
    return 2 * r * math.asin(math.sqrt(a))
# COMMAND ----------
def norm(x: Optional[str]) -> str:
    return (x or "").strip().lower()
# COMMAND ----------
def map_term_to_concepts(text: Optional[str]) -> List[str]:
    """
    Suggested simple synonym-based mapper.
    Replace later with richer ontology / embeddings if needed.
    """
    if not text:
        return []

    text_l = text.lower()
    matches = []
    for row in capability_rows:
        concept_name = (row["concept_name"] or "").lower()
        synonyms = [s.lower() for s in (row["synonyms"] or [])]
        if concept_name in text_l or any(s in text_l for s in synonyms):
            matches.append(row["concept_id"])
    return sorted(list(set(matches)))
# COMMAND ----------
def get_tags(code: str) -> List[str]:
    return contradiction_tags.get(code, [])
# COMMAND ----------
def build_contradiction(
    facility_id: str,
    code: str,
    confidence: float,
    explanation: str,
    recommended_action: str,
    concept_ids: Optional[List[str]] = None,
    severity: Optional[str] = None,
) -> Dict[str, Any]:
    ontology = contradiction_ontology[code]
    return {
        "facility_id": facility_id,
        "contradiction_id": str(uuid.uuid4()),
        "contradiction_code": code,
        "contradiction_class": ontology["contradiction_class"],
        "severity": severity or ontology["severity_default"],
        "confidence": confidence,
        "concept_ids": concept_ids or [],
        "tags": get_tags(code),
        "explanation": explanation,
        "recommended_action": recommended_action,
    }
# COMMAND ----------
def build_evidence(
    facility_id: str,
    contradiction_id: str,
    evidence_type: str,
    evidence_field: Optional[str],
    evidence_value: Optional[str],
    evidence_quote: Optional[str],
    source_ref: Optional[str] = None,
) -> Dict[str, Any]:
    return {
        "facility_id": facility_id,
        "contradiction_id": contradiction_id,
        "evidence_type": evidence_type,
        "evidence_field": evidence_field,
        "evidence_value": evidence_value,
        "evidence_quote": evidence_quote,
        "source_ref": source_ref,
    }

# COMMAND ----------
def run_rule_checks(facility: Dict[str, Any], pincode_lookup: Optional[Dict[str, Any]]) -> Dict[str, List[Dict[str, Any]]]:
    facility_id = facility["unique_id"]
    contradictions = []
    evidence = []

    state = norm(facility.get("address_stateOrRegion"))
    postcode = str(facility.get("address_zipOrPostcode") or "").strip()
    lat = facility.get("latitude")
    lon = facility.get("longitude")
    year_est = str(facility.get("yearEstablished") or "").strip()

    # HARD: invalid year
    if year_est:
        if not year_est.isdigit() or int(year_est) > CURRENT_YEAR:
            c = build_contradiction(
                facility_id,
                "INVALID_YEAR",
                0.98,
                f"yearEstablished='{year_est}' is not a valid historical year.",
                "Correct yearEstablished or set to null if unknown."
            )
            contradictions.append(c)
            evidence.append(build_evidence(
                facility_id, c["contradiction_id"],
                "STRUCTURED_FIELD", "yearEstablished", year_est, None
            ))

    # HARD: postcode-state mismatch
    if postcode and pincode_lookup:
        pincode_state = norm(pincode_lookup.get("statename"))
        if state and pincode_state and state != pincode_state:
            c = build_contradiction(
                facility_id,
                "GEO_STATE_MISMATCH",
                0.99,
                f"address_stateOrRegion='{facility.get('address_stateOrRegion')}' conflicts with postcode state '{pincode_lookup.get('statename')}'.",
                "Verify the facility address and postcode."
            )
            contradictions.append(c)
            evidence.extend([
                build_evidence(facility_id, c["contradiction_id"], "STRUCTURED_FIELD", "address_stateOrRegion", facility.get("address_stateOrRegion"), None),
                build_evidence(facility_id, c["contradiction_id"], "LOOKUP_MATCH", "statename", pincode_lookup.get("statename"), None, "india_post_pincode_directory"),
                build_evidence(facility_id, c["contradiction_id"], "LOOKUP_MATCH", "pincode", str(pincode_lookup.get("pincode")), None, "india_post_pincode_directory"),
            ])

    # HARD: coordinates too far from postcode coordinates
    if postcode and pincode_lookup and lat is not None and lon is not None:
        try:
            p_lat = float(pincode_lookup.get("latitude")) if pincode_lookup.get("latitude") is not None else None
            p_lon = float(pincode_lookup.get("longitude")) if pincode_lookup.get("longitude") is not None else None
            dist = haversine_km(lat, lon, p_lat, p_lon)
            if dist is not None and dist > DISTANCE_THRESHOLD_KM:
                c = build_contradiction(
                    facility_id,
                    "GEO_POSTCODE_COORD_MISMATCH",
                    0.95,
                    f"Facility coordinates are approximately {dist:.1f} km from postcode coordinates.",
                    "Verify coordinates and postcode."
                )
                contradictions.append(c)
                evidence.extend([
                    build_evidence(facility_id, c["contradiction_id"], "STRUCTURED_FIELD", "latitude", str(lat), None),
                    build_evidence(facility_id, c["contradiction_id"], "STRUCTURED_FIELD", "longitude", str(lon), None),
                    build_evidence(facility_id, c["contradiction_id"], "LOOKUP_MATCH", "pincode_latitude", str(p_lat), None, "india_post_pincode_directory"),
                    build_evidence(facility_id, c["contradiction_id"], "LOOKUP_MATCH", "pincode_longitude", str(p_lon), None, "india_post_pincode_directory"),
                ])
        except Exception:
            pass

    # LOGICAL: very rough staffing/capability mismatch initial screen
    number_doctors = str(facility.get("numberDoctors") or "").strip().lower()
    capacity = str(facility.get("capacity") or "").strip().lower()
    capabilities_raw = " ".join([
        str(facility.get("specialties") or ""),
        str(facility.get("procedure") or ""),
        str(facility.get("equipment") or ""),
        str(facility.get("capability") or ""),
        str(facility.get("description") or "")
    ]).lower()

    advanced_terms = ["icu", "surgery", "operating theatre", "oncology", "chemotherapy"]
    if any(t in capabilities_raw for t in advanced_terms):
        if number_doctors in ["", "0", "unknown"] or capacity in ["", "0", "unknown"]:
            c = build_contradiction(
                facility_id,
                "STAFFING_CAPABILITY_MISMATCH",
                0.60,
                "Advanced capability terms appear, but staffing/capacity fields are missing or weak.",
                "Review numberDoctors and capacity against the claimed services."
            )
            contradictions.append(c)
            evidence.extend([
                build_evidence(facility_id, c["contradiction_id"], "STRUCTURED_FIELD", "numberDoctors", str(facility.get("numberDoctors")), None),
                build_evidence(facility_id, c["contradiction_id"], "STRUCTURED_FIELD", "capacity", str(facility.get("capacity")), None),
            ])

    return {"contradictions": contradictions, "evidence": evidence}
# COMMAND ----------
facilities_df = spark.table(RAW_FACILITY_TABLE)

pincode_df = spark.table(PINCODE_TABLE).select(
    F.col("pincode").cast("string").alias("postcode_key"),
    "pincode", "district", "statename", "latitude", "longitude"
)

# COMMAND ----------
def get_facility_batch(limit: int = MAX_BATCH_SIZE):
    """
    Suggested queue behaviour:
    - If queue table has pending rows, process those
    - Else fall back to raw facilities
    """
    if spark.catalog.tableExists(QUEUE_TABLE):
        pending = spark.table(QUEUE_TABLE).filter(F.col("status") == "PENDING").limit(limit)
        pending_ids = [r["facility_id"] for r in pending.collect()]
        if pending_ids:
            return facilities_df.filter(F.col("unique_id").isin(pending_ids))
    return facilities_df.limit(limit)
# COMMAND ----------
def build_agent_payload(facility: Dict[str, Any], rule_findings: Dict[str, Any]) -> Dict[str, Any]:
    return {
        "facility": {
            "unique_id": facility.get("unique_id"),
            "name": facility.get("name"),
            "organization_type": facility.get("organization_type"),
            "yearEstablished": facility.get("yearEstablished"),
            "address_city": facility.get("address_city"),
            "address_stateOrRegion": facility.get("address_stateOrRegion"),
            "address_zipOrPostcode": facility.get("address_zipOrPostcode"),
            "latitude": facility.get("latitude"),
            "longitude": facility.get("longitude"),
            "specialties": facility.get("specialties"),
            "procedure": facility.get("procedure"),
            "equipment": facility.get("equipment"),
            "capability": facility.get("capability"),
            "numberDoctors": facility.get("numberDoctors"),
            "capacity": facility.get("capacity"),
            "description": facility.get("description"),
            "source_urls": facility.get("source_urls"),
        },
        "rule_findings": rule_findings["contradictions"],
        "ontology_classes": ["HARD", "LOGICAL", "WEAK_EVIDENCE"],
        "allowed_codes": list(contradiction_ontology.keys()),
    }
# COMMAND ----------
SYSTEM_PROMPT = """
You are a governed healthcare data quality agent.

Task:
Analyse one healthcare facility record and detect contradictions.

Requirements:
1. Use only the provided facility fields and prior rule findings.
2. Map every contradiction to one allowed contradiction_code.
3. Every contradiction must keep one contradiction_class:
   HARD, LOGICAL, or WEAK_EVIDENCE.
4. Map healthcare terms to canonical concept IDs where possible.
5. Always include evidence:
   - structured field references
   - exact text quotes from description when available
6. If evidence is weak, say so explicitly.
7. Return valid JSON only.

Return shape:
{
  "capability_assertions": [
    {
      "raw_term": "string",
      "mapped_concept_ids": ["string"],
      "evidence_quote": "string",
      "evidence_strength": "STRONG|WEAK",
      "confidence": 0.0
    }
  ],
  "contradictions": [
    {
      "contradiction_code": "string",
      "contradiction_class": "HARD|LOGICAL|WEAK_EVIDENCE",
      "severity": "HIGH|MEDIUM|LOW",
      "confidence": 0.0,
      "concept_ids": ["string"],
      "explanation": "string",
      "recommended_action": "string",
      "evidence": [
        {
          "evidence_type": "STRUCTURED_FIELD|DESCRIPTION_QUOTE|SOURCE_URL",
          "evidence_field": "string",
          "evidence_value": "string",
          "evidence_quote": "string",
          "source_ref": "string"
        }
      ]
    }
  ]
}
"""
# COMMAND ----------
import requests
import os
import json

# ✅ Your endpoint
DATABRICKS_HOST = "https://dbc-bbfb0194-f4b4.cloud.databricks.com"
ENDPOINT_NAME = "openai-llm"

def call_contradiction_agent(payload):

    url = f"{DATABRICKS_HOST}/serving-endpoints/{ENDPOINT_NAME}/invocations"

    # ✅ Get Databricks token (works inside notebook)
    token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()

    headers = {
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/json",
    }

    # ✅ Proper LLM chat payload
    request_body = {
        "messages": [
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": json.dumps(payload)
            }
        ]
        # Note: temperature parameter removed - this model only supports default (1.0)
    }

    response = requests.post(url, headers=headers, json=request_body)

    if response.status_code != 200:
        raise Exception(f"Model Serving error: {response.text}")

    # ✅ Parse response safely
    result = response.json()

    try:
        # Most Databricks LLM endpoints follow this structure
        content = result["choices"][0]["message"]["content"]
        
        # ✅ Convert string → JSON
        parsed_output = json.loads(content)
    
    except Exception as e:
        raise Exception(f"Failed to parse model output: {result}")

    return parsed_output
# Testing for one payload
# COMMAND ----------
def validate_and_normalise_agent_output(facility_id: str, agent_output: Dict[str, Any]) -> Dict[str, Any]:
    valid_contradictions = []
    valid_evidence = []

    for item in agent_output.get("contradictions", []):
        code = item.get("contradiction_code")
        if code not in contradiction_ontology:
            continue

        ontology_info = contradiction_ontology[code]
        contradiction_class = item.get("contradiction_class") or ontology_info["contradiction_class"]
        if contradiction_class not in ["HARD", "LOGICAL", "WEAK_EVIDENCE"]:
            contradiction_class = ontology_info["contradiction_class"]

        concept_ids = item.get("concept_ids") or []
        concept_ids = [c for c in concept_ids if any(r["concept_id"] == c for r in capability_rows)]

        contradiction_id = str(uuid.uuid4())

        detail = {
            "facility_id": facility_id,
            "contradiction_id": contradiction_id,
            "contradiction_code": code,
            "contradiction_class": contradiction_class,
            "severity": item.get("severity") or ontology_info["severity_default"],
            "confidence": float(item.get("confidence") or 0.5),
            "concept_ids": concept_ids,
            "tags": get_tags(code),
            "explanation": item.get("explanation"),
            "recommended_action": item.get("recommended_action"),
        }
        valid_contradictions.append(detail)

        for ev in item.get("evidence", []):
            valid_evidence.append({
                "facility_id": facility_id,
                "contradiction_id": contradiction_id,
                "evidence_type": ev.get("evidence_type"),
                "evidence_field": ev.get("evidence_field"),
                "evidence_value": ev.get("evidence_value"),
                "evidence_quote": ev.get("evidence_quote"),
                "source_ref": ev.get("source_ref"),
            })

    return {
        "contradictions": valid_contradictions,
        "evidence": valid_evidence
    }
# COMMAND ----------
def process_one_facility(facility_row: Dict[str, Any], pincode_map: Dict[str, Dict[str, Any]]) -> Dict[str, Any]:
    facility_id = facility_row["unique_id"]
    run_id = str(uuid.uuid4())
    run_ts = datetime.utcnow()

    try:
        postcode_key = str(facility_row.get("address_zipOrPostcode") or "").strip()
        pincode_lookup = pincode_map.get(postcode_key)

        # 1. Deterministic rules
        rule_findings = run_rule_checks(facility_row, pincode_lookup)

        # 2. Agent payload
        payload = build_agent_payload(facility_row, rule_findings)

        # 3. Call agent
        raw_agent_output = call_contradiction_agent(payload)

        # 4. Validate / normalise model output
        norm_agent_output = validate_and_normalise_agent_output(facility_id, raw_agent_output)

        # 5. Combine rule + agent results
        all_contradictions = rule_findings["contradictions"] + norm_agent_output["contradictions"]
        all_evidence = rule_findings["evidence"] + norm_agent_output["evidence"]

        # 6. Counts by class
        hard_count = sum(1 for c in all_contradictions if c["contradiction_class"] == "HARD")
        logical_count = sum(1 for c in all_contradictions if c["contradiction_class"] == "LOGICAL")
        weak_count = sum(1 for c in all_contradictions if c["contradiction_class"] == "WEAK_EVIDENCE")

        # 7. Suggested score
        risk_score = min(
            1.0,
            hard_count * 0.35 + logical_count * 0.20 + weak_count * 0.10
        )

        if risk_score >= 0.80:
            review_priority = "HIGH"
        elif risk_score >= 0.40:
            review_priority = "MEDIUM"
        else:
            review_priority = "LOW"

        summary = {
            "run_id": run_id,
            "facility_id": facility_id,
            "run_ts": run_ts,
            "agent_version": AGENT_VERSION,
            "hard_count": hard_count,
            "logical_count": logical_count,
            "weak_evidence_count": weak_count,
            "overall_risk_score": risk_score,
            "review_priority": review_priority,
            "ontology_version": ONTOLOGY_VERSION,
            "processing_status": "SUCCESS",
            "error_message": None
        }

        return {
            "summary": summary,
            "contradictions": [
                {**c, "run_id": run_id, "created_ts": run_ts} for c in all_contradictions
            ],
            "evidence": [
                {**e, "run_id": run_id, "created_ts": run_ts} for e in all_evidence
            ]
        }

    except Exception as e:
        return {
            "summary": {
                "run_id": run_id,
                "facility_id": facility_id,
                "run_ts": run_ts,
                "agent_version": AGENT_VERSION,
                "hard_count": 0,
                "logical_count": 0,
                "weak_evidence_count": 0,
                "overall_risk_score": 0.0,
                "review_priority": "LOW",
                "ontology_version": ONTOLOGY_VERSION,
                "processing_status": "FAILED",
                "error_message": str(e)
            },
            "contradictions": [],
            "evidence": []
        }
# COMMAND ----------
summary_schema = T.StructType([
    T.StructField("run_id", T.StringType()),
    T.StructField("facility_id", T.StringType()),
    T.StructField("run_ts", T.TimestampType()),
    T.StructField("agent_version", T.StringType()),
    T.StructField("hard_count", T.IntegerType()),
    T.StructField("logical_count", T.IntegerType()),
    T.StructField("weak_evidence_count", T.IntegerType()),
    T.StructField("overall_risk_score", T.DoubleType()),
    T.StructField("review_priority", T.StringType()),
    T.StructField("ontology_version", T.StringType()),
    T.StructField("processing_status", T.StringType()),
    T.StructField("error_message", T.StringType()),
])

detail_schema = T.StructType([
    T.StructField("run_id", T.StringType()),
    T.StructField("facility_id", T.StringType()),
    T.StructField("contradiction_id", T.StringType()),
    T.StructField("contradiction_code", T.StringType()),
    T.StructField("contradiction_class", T.StringType()),
    T.StructField("severity", T.StringType()),
    T.StructField("confidence", T.DoubleType()),
    T.StructField("concept_ids", T.ArrayType(T.StringType())),
    T.StructField("tags", T.ArrayType(T.StringType())),
    T.StructField("explanation", T.StringType()),
    T.StructField("recommended_action", T.StringType()),
    T.StructField("created_ts", T.TimestampType()),
])

evidence_schema = T.StructType([
    T.StructField("run_id", T.StringType()),
    T.StructField("facility_id", T.StringType()),
    T.StructField("contradiction_id", T.StringType()),
    T.StructField("evidence_type", T.StringType()),
    T.StructField("evidence_field", T.StringType()),
    T.StructField("evidence_value", T.StringType()),
    T.StructField("evidence_quote", T.StringType()),
    T.StructField("source_ref", T.StringType()),
    T.StructField("created_ts", T.TimestampType()),
])
# COMMAND ----------
def write_batch_results(batch_results: List[Dict[str, Any]]):
    summaries = [r["summary"] for r in batch_results]
    contradictions = [d for r in batch_results for d in r["contradictions"]]
    evidence = [e for r in batch_results for e in r["evidence"]]

    if summaries:
        spark.createDataFrame(summaries, summary_schema).write.mode("append").saveAsTable(RUN_SUMMARY_TABLE)

    if contradictions:
        spark.createDataFrame(contradictions, detail_schema).write.mode("append").saveAsTable(DETAIL_TABLE)

    if evidence:
        spark.createDataFrame(evidence, evidence_schema).write.mode("append").saveAsTable(EVIDENCE_TABLE)
# COMMAND ----------
def mark_queue_status(facility_ids: List[str], status: str, run_id: Optional[str] = None):
    if not facility_ids:
        return

    updates = [(fid, status, run_id, datetime.utcnow()) for fid in facility_ids]
    schema = T.StructType([
        T.StructField("facility_id", T.StringType()),
        T.StructField("status", T.StringType()),
        T.StructField("last_run_id", T.StringType()),
        T.StructField("updated_ts", T.TimestampType()),
    ])

    tmp_name = f"{CATALOG}.{ORCH_SCHEMA}.__queue_updates_tmp"
    spark.createDataFrame(updates, schema).write.mode("overwrite").saveAsTable(tmp_name)

    spark.sql(f"""
    MERGE INTO {QUEUE_TABLE} t
    USING {tmp_name} s
    ON t.facility_id = s.facility_id
    WHEN MATCHED THEN UPDATE SET
      t.status = s.status,
      t.last_run_id = s.last_run_id,
      t.updated_ts = s.updated_ts,
      t.attempts = COALESCE(t.attempts, 0) + 1
    WHEN NOT MATCHED THEN INSERT (
      facility_id, status, last_run_id, attempts, updated_ts
    ) VALUES (
      s.facility_id, s.status, s.last_run_id, 1, s.updated_ts
    )
    """)
# COMMAND ----------
batch_df = get_facility_batch(MAX_BATCH_SIZE)
facility_rows = [r.asDict() for r in batch_df.collect()]

postcode_keys = list({
    str(r.get("address_zipOrPostcode") or "").strip()
    for r in facility_rows
    if str(r.get("address_zipOrPostcode") or "").strip()
})

pincode_map_df = pincode_df.filter(F.col("postcode_key").isin(postcode_keys))
pincode_map = {r["postcode_key"]: r.asDict() for r in pincode_map_df.collect()}
# COMMAND ----------
facility_ids = [r["unique_id"] for r in facility_rows]
mark_queue_status(facility_ids, "RUNNING")
# COMMAND ----------
batch_results = []
for facility in facility_rows:
    result = process_one_facility(facility, pincode_map)
    batch_results.append(result)
# COMMAND ----------
write_batch_results(batch_results)
# COMMAND ----------
success_ids = [r["summary"]["facility_id"] for r in batch_results if r["summary"]["processing_status"] == "SUCCESS"]
failed_ids  = [r["summary"]["facility_id"] for r in batch_results if r["summary"]["processing_status"] == "FAILED"]

mark_queue_status(success_ids, "SUCCESS")
mark_queue_status(failed_ids, "FAILED")
# COMMAND ----------
spark.sql(f"""
SELECT
  contradiction_class,
  count(*) AS contradiction_count
FROM {DETAIL_TABLE}
GROUP BY contradiction_class
ORDER BY contradiction_count DESC
""").display()
# COMMAND ----------
spark.sql(f"""
SELECT
  facility_id,
  run_ts,
  hard_count,
  logical_count,
  weak_evidence_count,
  overall_risk_score,
  review_priority
FROM {RUN_SUMMARY_TABLE}
WHERE processing_status = 'SUCCESS'
ORDER BY overall_risk_score DESC, run_ts DESC
LIMIT 100
""").display()